# ✈️ Airline Behavioral Intelligence — Full Pipeline (Google Colab)

**Run this notebook top-to-bottom to execute the complete 7-stage pipeline.**

### What this notebook does:
| Step | Action |
|------|--------|
| 1 | Install all dependencies |
| 2 | Clone project from GitHub |
| 3 | Upload your data files |
| 4 | Run Stages 2 → 7 (cleaning → retention engine) |
| 5 | Display model results, segment breakdown, revenue at risk |
| 6 | Download all outputs (models, figures, predictions) |

### Required data files (upload when prompted):
- `Customer Loyalty History.csv`
- `Customer Flight Activity.csv`

---

## ⚙️ SECTION 1 — Setup

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────────
print('Installing packages... (takes ~60 seconds)')
import subprocess, sys

packages = [
    'pandas==2.0.3',
    'numpy==1.24.3',
    'scikit-learn==1.3.0',
    'xgboost==1.7.6',
    'imbalanced-learn==0.11.0',
    'matplotlib==3.7.2',
    'seaborn==0.12.2',
    'plotly==5.15.0',
    'streamlit==1.25.0',
    'joblib==1.3.1',
    'shap==0.42.1',
]

for pkg in packages:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True, text=True
    )
    status = '✓' if result.returncode == 0 else '✗'
    print(f'  {status} {pkg}')

print('\n✅ All packages installed.')

In [ ]:
# ── Clone project from GitHub ────────────────────────────────────────────────
import os

REPO_URL  = 'https://github.com/Arjun-Gangwar1/Airline-Loyalty-Programs.git'
PROJ_DIR  = '/content/airline_project'

if os.path.exists(PROJ_DIR):
    print('Project already cloned. Pulling latest...')
    os.system(f'cd {PROJ_DIR} && git pull -q')
else:
    print('Cloning project from GitHub...')
    result = os.system(f'git clone -q {REPO_URL} {PROJ_DIR}')
    if result == 0:
        print(f'✅ Cloned to {PROJ_DIR}')
    else:
        print('❌ Clone failed — check the repo URL')

# Set working directory and Python path
os.chdir(PROJ_DIR)
if PROJ_DIR not in sys.path:
    sys.path.insert(0, PROJ_DIR)

print(f'Working directory: {os.getcwd()}')
print('Files:', os.listdir('.'))

In [ ]:
# ── Create directory structure ────────────────────────────────────────────────
from pathlib import Path

dirs = [
    'data/raw', 'data/processed', 'data/final',
    'outputs/models', 'outputs/figures/exploration',
    'outputs/figures/churn', 'outputs/figures/features',
    'outputs/figures/segmentation', 'outputs/figures/models',
    'outputs/figures/retention', 'outputs/reports',
    'checkpoints', 'logs',
]
for d in dirs:
    Path(d).mkdir(parents=True, exist_ok=True)

# Initialise progress checkpoint
import json
progress = {'current_stage': 0, 'completed_stages': [], 'last_updated': ''}
with open('checkpoints/progress.json', 'w') as f:
    json.dump(progress, f, indent=2)

print('✅ Directory structure ready.')

In [ ]:
# ── Upload data files ─────────────────────────────────────────────────────────
#
# Upload BOTH files when the dialog appears:
#   • Customer Loyalty History.csv
#   • Customer Flight Activity.csv
#
from google.colab import files
import shutil

print('📂 Select your data files (hold Ctrl/Cmd to select multiple)...')
uploaded = files.upload()

for filename, content in uploaded.items():
    dest = Path('data/raw') / filename
    dest.write_bytes(content)
    print(f'  ✓ Saved: {dest}  ({len(content)/1024:.0f} KB)')

# Verify
raw_files = list(Path('data/raw').glob('*.csv'))
print(f'\n✅ {len(raw_files)} file(s) in data/raw/:')
for f in raw_files:
    print(f'   • {f.name}')

---
## 🔄 SECTION 2 — Run the Pipeline

In [ ]:
# ── Helper: run a stage script ────────────────────────────────────────────────
import subprocess, time

def run_stage(script_name, stage_num, stage_label):
    print(f'\n{"="*60}')
    print(f'  STAGE {stage_num}: {stage_label}')
    print(f'{"="*60}')
    start = time.time()
    result = subprocess.run(
        [sys.executable, script_name],
        capture_output=True, text=True, cwd=PROJ_DIR
    )
    elapsed = time.time() - start
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr[-2000:] if result.stderr else '')
        raise RuntimeError(f'Stage {stage_num} failed. See output above.')
    print(f'  ✅ Stage {stage_num} complete in {elapsed:.1f}s')
    return result

In [ ]:
# ── Stage 2: Data Cleaning ────────────────────────────────────────────────────
run_stage('2_data_cleaning.py', 2, 'DATA CLEANING')

In [ ]:
# ── Stage 3: Churn Definition ─────────────────────────────────────────────────
run_stage('3_churn_definition.py', 3, 'CHURN DEFINITION')

In [ ]:
# ── Stage 4: Feature Engineering ─────────────────────────────────────────────
run_stage('4_feature_engineering.py', 4, 'FEATURE ENGINEERING')

In [ ]:
# ── Stage 5: Customer Segmentation ───────────────────────────────────────────
run_stage('5_customer_segmentation.py', 5, 'CUSTOMER SEGMENTATION')

In [ ]:
# ── Stage 6: Baseline Models ──────────────────────────────────────────────────
run_stage('6_baseline_models.py', 6, 'BASELINE MODELS')

In [ ]:
# ── Stage 7: Retention Engine ─────────────────────────────────────────────────
run_stage('7_retention_engine.py', 7, 'RETENTION ENGINE')

---
## 📊 SECTION 3 — Results & Visualizations

In [ ]:
# ── Model performance summary ─────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import warnings
warnings.filterwarnings('ignore')

reports = Path('outputs/reports')
final   = Path('data/final')

print('='*60)
print('  MODEL PERFORMANCE')
print('='*60)

model_comp_path = reports / 'model_comparison.csv'
if model_comp_path.exists():
    model_df = pd.read_csv(model_comp_path)
    display_cols = [c for c in ['Model','ROC_AUC','Recall','Precision','F1','CV_AUC_mean']
                    if c in model_df.columns]
    print(model_df[display_cols].to_string(index=False))

    best = model_df.loc[model_df['ROC_AUC'].idxmax()]
    print(f'\n🏆 Best model : {best["Model"]}')
    print(f'   AUC        : {best["ROC_AUC"]:.4f}')
    print(f'   Recall     : {best["Recall"]:.4f}  ← catches {best["Recall"]*100:.0f}% of churners')
    print(f'   Precision  : {best["Precision"]:.4f}  ← {best["Precision"]*100:.0f}% of alerts are correct')
else:
    print('model_comparison.csv not found — Stage 6 may not have completed.')

In [ ]:
# ── Churn & prediction summary ────────────────────────────────────────────────
print('='*60)
print('  CHURN PREDICTION SUMMARY')
print('='*60)

preds_path = final / 'all_customer_predictions.csv'
if not preds_path.exists():
    preds_path = final / 'churn_predictions.csv'

if preds_path.exists():
    preds = pd.read_csv(preds_path)
    total = len(preds)

    churn_col = 'predicted_churn' if 'predicted_churn' in preds.columns else 'churn'
    prob_col  = 'churn_probability' if 'churn_probability' in preds.columns else None

    at_risk = preds[churn_col].sum() if churn_col in preds.columns else 0

    print(f'  Total customers    : {total:,}')
    print(f'  Predicted at-risk  : {at_risk:,}  ({at_risk/total*100:.1f}%)')
    print(f'  Safe customers     : {total-at_risk:,}  ({(total-at_risk)/total*100:.1f}%)')

    if 'risk_level' in preds.columns:
        risk_counts = preds['risk_level'].value_counts()
        print(f'\n  Risk level breakdown:')
        for level in ['high','medium','low']:
            n = risk_counts.get(level, 0)
            print(f'    {level.upper():8s}: {n:,}  ({n/total*100:.1f}%)')
else:
    print('Predictions file not found.')

In [ ]:
# ── Customer segments summary ─────────────────────────────────────────────────
print('='*60)
print('  CUSTOMER SEGMENTS')
print('='*60)

seg_path = final / 'customer_features_segmented.csv'
if not seg_path.exists():
    seg_path = final / 'customer_segments.csv'

if seg_path.exists():
    seg_df = pd.read_csv(seg_path)
    seg_col = 'segment_name' if 'segment_name' in seg_df.columns else 'segment'

    if seg_col in seg_df.columns:
        seg_counts = seg_df[seg_col].value_counts()
        print(f'  {len(seg_counts)} segments found:\n')

        churn_col = next((c for c in ['churn','churned'] if c in seg_df.columns), None)
        clv_col   = 'clv' if 'clv' in seg_df.columns else None

        for seg, count in seg_counts.items():
            pct = count / len(seg_df) * 100
            extras = ''
            mask = seg_df[seg_col] == seg
            if churn_col:
                cr = seg_df.loc[mask, churn_col].mean() * 100
                extras += f'  churn={cr:.0f}%'
            if clv_col:
                mc = seg_df.loc[mask, clv_col].median()
                extras += f'  median_CLV=${mc:,.0f}'
            print(f'  {seg:<28} {count:5,} ({pct:.1f}%){extras}')
else:
    print('Segment file not found.')

In [ ]:
# ── Revenue at risk summary ───────────────────────────────────────────────────
print('='*60)
print('  RETENTION ENGINE — REVENUE IMPACT')
print('='*60)

ret_path = reports / 'retention_actions.csv'
if ret_path.exists():
    ret_df = pd.read_csv(ret_path)

    rev_col  = 'revenue_at_risk'  if 'revenue_at_risk'  in ret_df.columns else None
    save_col = 'potential_save'   if 'potential_save'   in ret_df.columns else None

    if rev_col:
        total_rev  = ret_df[rev_col].sum()
        total_save = ret_df[save_col].sum() if save_col else 0
        print(f'  Total revenue at risk : ${total_rev:,.0f}')
        print(f'  Potential savings     : ${total_save:,.0f}')
        print(f'  ROI on retention      : {total_save/max(total_rev,1)*100:.1f}%')

        if 'segment_name' in ret_df.columns:
            print(f'\n  Revenue at risk by segment:')
            seg_rev = ret_df.groupby('segment_name')[rev_col].sum().sort_values(ascending=False)
            for seg, val in seg_rev.items():
                print(f'    {seg:<28} ${val:>10,.0f}')
else:
    print('retention_actions.csv not found.')

In [ ]:
# ── Display saved figures ─────────────────────────────────────────────────────
from IPython.display import display, Image
import glob

figure_paths = sorted(glob.glob('outputs/figures/**/*.png', recursive=True))
print(f'Found {len(figure_paths)} figures. Displaying key ones...\n')

# Key figures to show
priority_keywords = [
    'roc_curves', 'feature_importance', 'segment_overview',
    'segments_pca', 'revenue_at_risk', 'churn_distribution',
    'feature_correlations', 'confusion',
]

shown = 0
for keyword in priority_keywords:
    matches = [p for p in figure_paths if keyword in p]
    if matches:
        print(f'--- {matches[0]} ---')
        display(Image(filename=matches[0], width=900))
        shown += 1

if shown == 0:
    print('No figures found yet — stages may not have completed.')
    print('Available figures:', figure_paths)

In [ ]:
# ── Feature importance table ──────────────────────────────────────────────────
feat_imp_path = reports / 'feature_importance.csv'
if feat_imp_path.exists():
    feat_imp = pd.read_csv(feat_imp_path)
    print('Top 15 Most Important Features for Churn Prediction:')
    print('='*50)
    print(feat_imp.head(15).to_string(index=False))
else:
    # Try to load from saved model and recompute
    import joblib
    model_path = Path('outputs/models/best_model.pkl')
    feat_path  = final / 'customer_features_segmented.csv'
    if not feat_path.exists():
        feat_path = final / 'customer_features.csv'

    if model_path.exists() and feat_path.exists():
        model   = joblib.load(model_path)
        feat_df = pd.read_csv(feat_path)
        drop_cols = {'loyalty_number','churn','churned','segment_name','segment',
                     'cluster','gender','education','marital_status','loyalty_card',
                     'enrollment_type','country','province','city','postal_code',
                     'last_activity_date','cancellation_year','cancellation_month'}
        feature_cols = [c for c in feat_df.columns
                        if c not in drop_cols and feat_df[c].dtype != object]

        if hasattr(model, 'feature_importances_'):
            imp = pd.Series(model.feature_importances_, index=feature_cols)
            imp = imp.sort_values(ascending=False).head(15)
            print('Top 15 Feature Importances (from best model):')
            print('='*45)
            for feat, val in imp.items():
                bar = '█' * int(val * 300)
                print(f'  {feat:<35} {val:.4f}  {bar}')
            imp.reset_index().rename(columns={'index':'feature', 0:'importance'}).to_csv(
                reports / 'feature_importance.csv', index=False
            )
            print('\n✓ Saved: outputs/reports/feature_importance.csv')
    else:
        print('Feature importance not available — run Stage 6 first.')

In [ ]:
# ── Top 10 priority customers for retention ───────────────────────────────────
priority_path = reports / 'priority_action_list.csv'
if not priority_path.exists():
    priority_path = reports / 'retention_actions.csv'

if priority_path.exists():
    priority_df = pd.read_csv(priority_path)
    print('TOP 10 PRIORITY CUSTOMERS FOR RETENTION')
    print('='*70)

    show_cols = [c for c in [
        'loyalty_number', 'segment_name', 'risk_level',
        'churn_probability', 'clv', 'revenue_at_risk',
        'recommended_action', 'offer', 'timing'
    ] if c in priority_df.columns]

    sort_col = 'priority_score' if 'priority_score' in priority_df.columns \
               else 'revenue_at_risk' if 'revenue_at_risk' in priority_df.columns \
               else show_cols[0]

    top10 = priority_df.nlargest(10, sort_col)[show_cols]

    pd.set_option('display.max_colwidth', 40)
    pd.set_option('display.width', 200)
    print(top10.to_string(index=False))
else:
    print('Priority action list not found.')

In [ ]:
# ── Save pipeline summary JSON ────────────────────────────────────────────────
from datetime import datetime

summary = {'run_date': datetime.now().isoformat(), 'status': 'complete'}

# Model metrics
if (reports / 'model_comparison.csv').exists():
    mc = pd.read_csv(reports / 'model_comparison.csv')
    for _, row in mc.iterrows():
        name = row.get('Model', row.get('model', 'unknown'))
        summary[f'{name}_auc'] = float(row.get('ROC_AUC', row.get('roc_auc', 0)))

# Churn stats
if (final / 'all_customer_predictions.csv').exists():
    p = pd.read_csv(final / 'all_customer_predictions.csv')
    churn_col = 'predicted_churn' if 'predicted_churn' in p.columns else 'churn'
    summary['total_customers']   = len(p)
    summary['predicted_churners'] = int(p[churn_col].sum())
    summary['churn_rate_pct']    = round(p[churn_col].mean() * 100, 2)

# Revenue
if (reports / 'retention_actions.csv').exists():
    r = pd.read_csv(reports / 'retention_actions.csv')
    if 'revenue_at_risk' in r.columns:
        summary['total_revenue_at_risk'] = round(float(r['revenue_at_risk'].sum()), 2)
    if 'potential_save' in r.columns:
        summary['total_potential_save']  = round(float(r['potential_save'].sum()), 2)

with open(reports / 'pipeline_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Pipeline Summary:')
print(json.dumps(summary, indent=2))
print('\n✅ Saved: outputs/reports/pipeline_summary.json')

---
## 📦 SECTION 4 — Download All Outputs

In [ ]:
# ── Zip and download everything ───────────────────────────────────────────────
import shutil
from google.colab import files

print('Packaging outputs...')

# Create a combined results folder
results_dir = Path('/content/airline_results')
if results_dir.exists():
    shutil.rmtree(results_dir)
results_dir.mkdir()

# Copy outputs, models, predictions, figures
folders_to_copy = [
    ('outputs',    results_dir / 'outputs'),
    ('data/final', results_dir / 'data_final'),
]
for src, dst in folders_to_copy:
    if Path(src).exists():
        shutil.copytree(src, dst)
        print(f'  ✓ Copied {src}')

# Zip it
zip_path = '/content/airline_outputs'
shutil.make_archive(zip_path, 'zip', results_dir)
zip_size = Path(zip_path + '.zip').stat().st_size / (1024 * 1024)
print(f'\n✅ Created airline_outputs.zip ({zip_size:.1f} MB)')

# Download
print('⬇️  Downloading...')
files.download(zip_path + '.zip')

In [ ]:
# ── [Optional] Download individual files ─────────────────────────────────────
# Uncomment whichever files you want individually

# files.download('outputs/models/best_model.pkl')
# files.download('outputs/models/xgboost.pkl')
# files.download('data/final/all_customer_predictions.csv')
# files.download('outputs/reports/retention_actions.csv')
# files.download('outputs/reports/model_comparison.csv')
# files.download('outputs/reports/pipeline_summary.json')
# files.download('outputs/figures/models/roc_curves.png')
# files.download('outputs/figures/models/feature_importance.png')
print('Uncomment any line above to download that file individually.')

In [ ]:
# ── [Optional] Save to Google Drive instead of downloading ────────────────────
# Uncomment the lines below if you prefer saving to Drive

# from google.colab import drive
# drive.mount('/content/drive')
#
# drive_dest = '/content/drive/MyDrive/AirlineProject'
# if Path(drive_dest).exists():
#     shutil.rmtree(drive_dest)
# shutil.copytree(str(results_dir), drive_dest)
# print(f'✅ Saved to Google Drive: {drive_dest}')

print('Uncomment the lines above to save outputs to your Google Drive instead.')

---
## ✅ Pipeline Complete

### What you now have in `airline_outputs.zip`:

```
airline_outputs/
├── outputs/
│   ├── models/
│   │   ├── best_model.pkl          ← Best churn prediction model
│   │   ├── xgboost.pkl
│   │   ├── random_forest.pkl
│   │   ├── logistic_regression.pkl
│   │   └── feature_scaler.pkl
│   ├── figures/
│   │   ├── models/roc_curves.png
│   │   ├── models/feature_importance.png
│   │   ├── segmentation/segments_pca.png
│   │   └── retention/revenue_at_risk.png
│   └── reports/
│       ├── model_comparison.csv
│       ├── retention_actions.csv
│       ├── priority_action_list.csv
│       ├── feature_importance.csv
│       ├── pipeline_summary.json
│       └── retention_playbook.json
└── data_final/
    ├── customer_features.csv
    ├── customer_features_segmented.csv
    ├── all_customer_predictions.csv
    ├── churn_predictions.csv
    └── customer_segments.csv
```

### Next steps:
1. Copy model metrics from `pipeline_summary.json` → update your README
2. Use figures from `outputs/figures/` in your project report
3. Share `retention_actions.csv` as the business deliverable